# Phase 1 — Tokenization & Text Preprocessing

![Tokenization Preprocessing Flowchart](outputs/tokenization_preprocessing_flowchart.png)
Welcome to the first phase of your NLP learning journey! This is the entry point to every NLP pipeline. Raw text is messy — it contains HTML tags, URLs, punctuation, inconsistent casing, extra spaces, and noise. Before any model can process text, you must clean it and split it into meaningful units called **tokens**. Every downstream phase depends on this phase being done correctly.

### Sequential Task Flow

![Phase 1 Flowchart](phase1_flowchart.png)

```
Raw Text → Data Cleaning → Word Tokenization → Sentence Tokenization
→ Stopword Removal → Stemming → Lemmatization → Full Pipeline Comparison
```

## Table of Contents

- [0. Setup & Imports](#0-setup--imports)
- [Task 1 — Data Cleaning](#task-1--data-cleaning)
- [Step 1 — Load raw text](#step-1--load-raw-text)
- [Step 2 — Lowercase the text](#step-2--lowercase-the-text)
- [Step 3 — Remove HTML tags](#step-3--remove-html-tags)
- [Step 4 — Remove URLs](#step-4--remove-urls)
- [Step 5 — Remove special characters and punctuation](#step-5--remove-special-characters-and-punctuation)
- [Step 6 — Remove extra whitespace](#step-6--remove-extra-whitespace)
- [Task 2 — Word Tokenization](#task-2--word-tokenization)
- [Step 1 — Whitespace splitting (scratch)](#step-1--whitespace-splitting-scratch)
- [Step 2 — NLTK word_tokenize](#step-2--nltk-word_tokenize)
- [Step 3 — spaCy tokenizer](#step-3--spacy-tokenizer)
- [Step 4 — Edge case comparison](#step-4--edge-case-comparison)
- [Task 3 — Sentence Tokenization](#task-3--sentence-tokenization)
- [Step 1 — Period splitting (scratch)](#step-1--period-splitting-scratch)
- [Step 2 — NLTK sent_tokenize](#step-2--nltk-sent_tokenize)
- [Step 3 — Longer document comparison](#step-3--longer-document-comparison)
- [Task 4 — Stopword Removal](#task-4--stopword-removal)
- [Step 1 — Inspect the stopword list](#step-1--inspect-the-stopword-list)
- [Step 2 — Remove stopwords from a sentence](#step-2--remove-stopwords-from-a-sentence)
- [Step 3 — Negation failure demo](#step-3--negation-failure-demo)
- [Step 4 — Custom stopword list](#step-4--custom-stopword-list)
- [Task 5 — Stemming](#task-5--stemming)
- [Step 1 — Porter Stemmer](#step-1--porter-stemmer)
- [Step 2 — Snowball Stemmer comparison](#step-2--snowball-stemmer-comparison)
- [Task 6 — Lemmatization](#task-6--lemmatization)
- [Step 1 — spaCy lemmatizer](#step-1--spacy-lemmatizer)
- [Step 2 — POS dependency demo](#step-2--pos-dependency-demo)
- [Task 7 — Full Pipeline Comparison](#task-7--full-pipeline-comparison)
- [Summary & Key Takeaways](#summary--key-takeaways)

## 0. Setup & Imports

Before we begin, let's import all necessary libraries and download the required NLTK data packages. We install everything once at the top so every cell below can assume these are available.

In [1]:
import re
import unicodedata

import nltk
import spacy
import pandas as pd
import numpy as np

# Download NLTK data packages (only needed once)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Load the spaCy English model
nlp = spacy.load("en_core_web_sm")

print("✅ All libraries imported and data downloaded successfully.")
print(f"   NLTK version : {nltk.__version__}")
print(f"   spaCy version: {spacy.__version__}")
print(f"   Pandas version: {pd.__version__}")

✅ All libraries imported and data downloaded successfully.
   NLTK version : 3.9.4
   spaCy version: 3.8.14
   Pandas version: 3.0.3


---
## Task 1 — Data Cleaning

![Data Cleaning Workflow](outputs/task1_data_cleaning.png)


**Objective:** Transform raw, noisy text into a clean, normalized string ready for tokenization.

Real-world text is rarely clean. It arrives with HTML markup, embedded URLs, inconsistent casing, punctuation noise, and erratic whitespace. Each of these must be handled before any NLP model can process the text reliably. In this task, we build a step-by-step cleaning pipeline and observe what each step does — and what it might accidentally destroy.

### Step 1 — Load raw text

> **What to do:** Collect 5–10 sentences of messy real-world text. Include at least one sentence with a URL, one with HTML tags, one with numbers, one with special characters, and one in ALL CAPS.
>
> **Display:** Print the raw text exactly as loaded — every piece of noise visible.
>
> **Infer:** Identify every type of noise present. This list becomes your cleaning checklist.

In [2]:
raw_sentences = [
    "Check out this AMAZING article at https://example.com/nlp-guide — it's mind-blowing!!!",
    "<p>Natural Language Processing</p> is a <b>subfield</b> of AI &amp; linguistics.",
    "The product costs $1,299.99 and has a 4.5★ rating (out of 5).",
    "I    love   NLP...  it's    so    COOL!!!   Right?!?!",
    "BREAKING NEWS: Python 3.12 released — performance improved by 25%!!!",
    "Email me at user@example.com or visit www.example.org for details.",
    "The café served crème brûlée — absolutely délicieux! 😊",
    "RT @nlp_fan: #MachineLearning is the future of #AI!! Check it out!!!"
]

print("=" * 80)
print("RAW TEXT (as loaded)")
print("=" * 80)
for i, sent in enumerate(raw_sentences, 1):
    print(f"  [{i}] {sent}")

# Identify noise types
print("\n" + "=" * 80)
print("NOISE INVENTORY")
print("=" * 80)
noise_types = [
    "URLs         → https://example.com/nlp-guide, www.example.org",
    "HTML tags    → <p>, </p>, <b>, </b>",
    "HTML entities→ &amp;",
    "ALL CAPS     → AMAZING, COOL, BREAKING NEWS",
    "Special chars→ ★, —, 😊, $, @, #",
    "Numbers      → $1,299.99, 4.5, 25%, 3.12",
    "Extra spaces → multiple consecutive spaces in sentence [4]",
    "Punctuation  → !!!, ?!?!, ...",
    "Email address→ user@example.com",
    "Mentions/tags→ @nlp_fan, #MachineLearning, #AI",
    "Unicode      → café, crème, brûlée, délicieux"
]
for n in noise_types:
    print(f"  • {n}")

RAW TEXT (as loaded)
  [1] Check out this AMAZING article at https://example.com/nlp-guide — it's mind-blowing!!!
  [2] <p>Natural Language Processing</p> is a <b>subfield</b> of AI &amp; linguistics.
  [3] The product costs $1,299.99 and has a 4.5★ rating (out of 5).
  [4] I    love   NLP...  it's    so    COOL!!!   Right?!?!
  [5] BREAKING NEWS: Python 3.12 released — performance improved by 25%!!!
  [6] Email me at user@example.com or visit www.example.org for details.
  [7] The café served crème brûlée — absolutely délicieux! 😊
  [8] RT @nlp_fan: #MachineLearning is the future of #AI!! Check it out!!!

NOISE INVENTORY
  • URLs         → https://example.com/nlp-guide, www.example.org
  • HTML tags    → <p>, </p>, <b>, </b>
  • HTML entities→ &amp;
  • ALL CAPS     → AMAZING, COOL, BREAKING NEWS
  • Special chars→ ★, —, 😊, $, @, #
  • Numbers      → $1,299.99, 4.5, 25%, 3.12
  • Extra spaces → multiple consecutive spaces in sentence [4]
  • Punctuation  → !!!, ?!?!, ...
  • Email add

### Step 2 — Lowercase the text

> **What to do:** Convert the entire text string to lowercase.
>
> **Display:** Print the original text and the lowercased version side by side.
>
> **Infer:** Observe that `NLP`, `nlp`, and `Nlp` now all look the same. Note that this is not always desirable — `US` (country) and `us` (pronoun) are now identical. Record this as a known limitation.

In [3]:
print(f"{'ORIGINAL':<55} | {'LOWERCASED'}")
print("-" * 120)
for sent in raw_sentences:
    lower = sent.lower()
    # Truncate for display if needed
    orig_display = sent[:52] + "..." if len(sent) > 55 else sent
    lower_display = lower[:52] + "..." if len(lower) > 55 else lower
    print(f"{orig_display:<55} | {lower_display}")

print("\n⚠️  LIMITATION: 'US' (country) and 'us' (pronoun) are now identical after lowercasing.")
print("    'NLP' and 'nlp' merge correctly, but 'IT' (company) and 'it' (pronoun) also merge.")
print("    → Lowercasing is a lossy operation. It helps most tasks but hurts entity recognition.")

ORIGINAL                                                | LOWERCASED
------------------------------------------------------------------------------------------------------------------------
Check out this AMAZING article at https://example.co... | check out this amazing article at https://example.co...
<p>Natural Language Processing</p> is a <b>subfield<... | <p>natural language processing</p> is a <b>subfield<...
The product costs $1,299.99 and has a 4.5★ rating (o... | the product costs $1,299.99 and has a 4.5★ rating (o...
I    love   NLP...  it's    so    COOL!!!   Right?!?!   | i    love   nlp...  it's    so    cool!!!   right?!?!
BREAKING NEWS: Python 3.12 released — performance im... | breaking news: python 3.12 released — performance im...
Email me at user@example.com or visit www.example.or... | email me at user@example.com or visit www.example.or...
The café served crème brûlée — absolutely délicieux! 😊  | the café served crème brûlée — absolutely délicieux! 😊
RT @nlp_fan: #M

### Step 3 — Remove HTML tags

> **What to do:** Use a regex pattern to strip anything that looks like an HTML tag (text enclosed in `< >`).
>
> **Display:** Print a before/after pair for a sentence that contained an HTML tag.
>
> **Infer:** Confirm that the tag structure is gone but the text content between tags is preserved.

In [4]:
import html as html_module

def remove_html_tags(text):
    """Remove HTML tags using regex and decode HTML entities."""
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Decode HTML entities like &amp; → &, &lt; → <
    text = html_module.unescape(text)
    return text

# Demonstrate on the HTML-heavy sentence
html_sentence = raw_sentences[1]
cleaned = remove_html_tags(html_sentence)

print("BEFORE:", html_sentence)
print("AFTER: ", cleaned)
print()
print("✅ HTML tags (<p>, </p>, <b>, </b>) removed.")
print("✅ HTML entity '&amp;' decoded to '&'.")
print("✅ Text content 'Natural Language Processing' and 'subfield' preserved.")

BEFORE: <p>Natural Language Processing</p> is a <b>subfield</b> of AI &amp; linguistics.
AFTER:  Natural Language Processing is a subfield of AI & linguistics.

✅ HTML tags (<p>, </p>, <b>, </b>) removed.
✅ HTML entity '&amp;' decoded to '&'.
✅ Text content 'Natural Language Processing' and 'subfield' preserved.


### Step 4 — Remove URLs

> **What to do:** Use a regex pattern to remove `http://`, `https://`, and `www.` URLs entirely.
>
> **Display:** Print before/after for a URL-containing sentence.
>
> **Infer:** Observe that URLs add no semantic meaning to most NLP tasks and removing them reduces vocabulary noise.

In [5]:
def remove_urls(text):
    """Remove URLs starting with http://, https://, or www."""
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    return text

# Demonstrate on URL-containing sentences
for idx in [0, 5]:
    original = raw_sentences[idx]
    cleaned = remove_urls(original)
    print(f"BEFORE: {original}")
    print(f"AFTER:  {cleaned.strip()}")
    print("-" * 80)

print("\n✅ URLs removed. They add no semantic meaning to most NLP tasks.")
print("   In some domains (e.g., web classification), URLs might carry signal — but this is rare.")

BEFORE: Check out this AMAZING article at https://example.com/nlp-guide — it's mind-blowing!!!
AFTER:  Check out this AMAZING article at  — it's mind-blowing!!!
--------------------------------------------------------------------------------
BEFORE: Email me at user@example.com or visit www.example.org for details.
AFTER:  Email me at user@example.com or visit  for details.
--------------------------------------------------------------------------------

✅ URLs removed. They add no semantic meaning to most NLP tasks.
   In some domains (e.g., web classification), URLs might carry signal — but this is rare.


### Step 5 — Remove special characters and punctuation

> **What to do:** Strip all characters that are not letters, digits, or spaces using a regex substitution.
>
> **Display:** Print the text before and after this step. Also print a sorted list of every unique character that was removed.
>
> **Infer:** Notice that punctuation like `!` and `?` carried some sentiment signal (exclamation marks in positive text). Record that this step may discard useful signals — a tradeoff to revisit in Phase 4.

In [6]:
def remove_special_chars(text):
    """Remove all characters that are not letters, digits, or spaces."""
    # Track what we're removing
    removed = set(c for c in text if not c.isalnum() and not c.isspace())
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return cleaned, removed

# Demonstrate on a few sentences
test_sentences = [raw_sentences[0], raw_sentences[2], raw_sentences[3]]
all_removed = set()

for sent in test_sentences:
    cleaned, removed = remove_special_chars(sent)
    all_removed.update(removed)
    print(f"BEFORE: {sent}")
    print(f"AFTER:  {cleaned}")
    print(f"REMOVED: {sorted(removed)}")
    print("-" * 80)

print(f"\nAll unique characters removed across examples: {sorted(all_removed)}")
print(f"Total unique special characters: {len(all_removed)}")
print()
print("⚠️  TRADEOFF: Punctuation like '!' and '?' carried sentiment signal.")
print("   'AMAZING!!!' has stronger positive sentiment than 'AMAZING'.")
print("   → We lose this signal. We will revisit this tradeoff in Phase 4 (Sentiment Analysis).")

BEFORE: Check out this AMAZING article at https://example.com/nlp-guide — it's mind-blowing!!!
AFTER:  Check out this AMAZING article at httpsexamplecomnlpguide  its mindblowing
REMOVED: ['!', "'", '-', '.', '/', ':', '—']
--------------------------------------------------------------------------------
BEFORE: The product costs $1,299.99 and has a 4.5★ rating (out of 5).
AFTER:  The product costs 129999 and has a 45 rating out of 5
REMOVED: ['$', '(', ')', ',', '.', '★']
--------------------------------------------------------------------------------
BEFORE: I    love   NLP...  it's    so    COOL!!!   Right?!?!
AFTER:  I    love   NLP  its    so    COOL   Right
REMOVED: ['!', "'", '.', '?']
--------------------------------------------------------------------------------

All unique characters removed across examples: ['!', '$', "'", '(', ')', ',', '-', '.', '/', ':', '?', '—', '★']
Total unique special characters: 13

⚠️  TRADEOFF: Punctuation like '!' and '?' carried sentiment signal.

### Step 6 — Remove extra whitespace

> **What to do:** Collapse multiple consecutive spaces, tabs, and newlines into a single space. Strip leading and trailing whitespace.
>
> **Display:** Print a final cleaned version of all sentences.
>
> **Infer:** The text should now be clean, lowercase, and uniform. Compare it to your original raw text. This cleaned version is what enters the tokenizer.

In [7]:
def full_clean(text):
    """Apply the complete cleaning pipeline."""
    text = text.lower()
    text = remove_html_tags(text)
    text = remove_urls(text)
    text = re.sub(r'@\w+', '', text)           # Remove mentions
    text = re.sub(r'#(\w+)', r'\1', text)      # Keep hashtag text, remove #
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip()    # Normalize whitespace
    return text

print("=" * 100)
print(f"{'RAW TEXT':<60} | CLEANED TEXT")
print("=" * 100)
for sent in raw_sentences:
    cleaned = full_clean(sent)
    orig_display = sent[:57] + "..." if len(sent) > 60 else sent
    print(f"{orig_display:<60} | {cleaned}")

print("\n✅ All text is now lowercase, tag-free, URL-free, and uniformly spaced.")
print("   This cleaned text is ready to enter the tokenizer in Task 2.")

RAW TEXT                                                     | CLEANED TEXT
Check out this AMAZING article at https://example.com/nlp... | check out this amazing article at its mindblowing
<p>Natural Language Processing</p> is a <b>subfield</b> o... | natural language processing is a subfield of ai linguistics
The product costs $1,299.99 and has a 4.5★ rating (out of... | the product costs 129999 and has a 45 rating out of 5
I    love   NLP...  it's    so    COOL!!!   Right?!?!        | i love nlp its so cool right
BREAKING NEWS: Python 3.12 released — performance improve... | breaking news python 312 released performance improved by 25
Email me at user@example.com or visit www.example.org for... | email me at usercom or visit for details
The café served crème brûlée — absolutely délicieux! 😊       | the caf served crme brle absolutely dlicieux
RT @nlp_fan: #MachineLearning is the future of #AI!! Chec... | rt machinelearning is the future of ai check it out

✅ All text is now lowercase

---
## Task 2 — Word Tokenization

![Word Tokenization Comparison](outputs/task2_word_tokenization.png)


**Objective:** Split a cleaned sentence into a list of individual word tokens.

Tokenization seems trivial — just split on spaces, right? This task reveals why it is not. Contractions (`it's`, `don't`), hyphenated compounds (`state-of-the-art`), and edge cases (`$1,000.00`, `...`) all demand different strategies. We compare three approaches: a naive whitespace split, NLTK's rule-based tokenizer, and spaCy's statistical tokenizer.

### Step 1 — Whitespace splitting (scratch)

> **What to do:** Split the cleaned sentence using Python's built-in `.split()` method.
>
> **Display:** Print the resulting list of tokens and its length.
>
> **Infer:** This works for clean text. Now test it on `"it's a state-of-the-art model"` — observe how contractions and hyphenated words are handled (or mishandled).

In [8]:
# Test on a clean sentence
clean_text = "natural language processing is a subfield of artificial intelligence"
ws_tokens = clean_text.split()

print(f"Input:  '{clean_text}'")
print(f"Tokens: {ws_tokens}")
print(f"Count:  {len(ws_tokens)}")

# Now test on tricky cases (using original uncleaned text to see the problem)
tricky = "it's a state-of-the-art model"
tricky_tokens = tricky.split()

print(f"\nInput:  '{tricky}'")
print(f"Tokens: {tricky_tokens}")
print(f"Count:  {len(tricky_tokens)}")
print()
print("⚠️  'it's' stays as one token — should 'it' and "'s'"be separate?")
print("⚠️  'state-of-the-art' stays as one token — is this a single word or four?")
print("   → Whitespace splitting has no opinion on these. It only sees spaces.")

Input:  'natural language processing is a subfield of artificial intelligence'
Tokens: ['natural', 'language', 'processing', 'is', 'a', 'subfield', 'of', 'artificial', 'intelligence']
Count:  9

Input:  'it's a state-of-the-art model'
Tokens: ["it's", 'a', 'state-of-the-art', 'model']
Count:  4

⚠️  'it's' stays as one token — should 'it' and sbe separate?
⚠️  'state-of-the-art' stays as one token — is this a single word or four?
   → Whitespace splitting has no opinion on these. It only sees spaces.


### Step 2 — NLTK word_tokenize

> **What to do:** Apply NLTK's `word_tokenize` function to the same sentences.
>
> **Display:** Print the token list side by side with the whitespace-split version from Step 1.
>
> **Infer:** NLTK correctly splits `it's` into `it` and `'s`, and handles punctuation as separate tokens. Count how many tokens differ between the two approaches.

In [9]:
from nltk.tokenize import word_tokenize

# Same tricky sentence
tricky = "it's a state-of-the-art model"
ws_tokens = tricky.split()
nltk_tokens = word_tokenize(tricky)

print(f"Input: '{tricky}'\n")
print(f"{'Method':<20} | {'Token Count':<12} | Tokens")
print("-" * 80)
print(f"{'Whitespace':<20} | {len(ws_tokens):<12} | {ws_tokens}")
print(f"{'NLTK':<20} | {len(nltk_tokens):<12} | {nltk_tokens}")

# More examples
more_examples = [
    "NLP is state-of-the-art! However, it doesn't always work perfectly.",
    "The U.S.A. has a GDP of $21.43 trillion.",
    "I can't believe it's not butter!"
]

print("\n" + "=" * 90)
for sent in more_examples:
    ws = sent.split()
    nk = word_tokenize(sent)
    print(f"\nInput: '{sent}'")
    print(f"  Whitespace ({len(ws):>2} tokens): {ws}")
    print(f"  NLTK       ({len(nk):>2} tokens): {nk}")

print("\n✅ NLTK splits contractions: it's → [it, 's], doesn't → [does, n't], can't → [ca, n't]")
print("   This is Penn Treebank style — it isolates the negation particle 'n't'.")

Input: 'it's a state-of-the-art model'

Method               | Token Count  | Tokens
--------------------------------------------------------------------------------
Whitespace           | 4            | ["it's", 'a', 'state-of-the-art', 'model']
NLTK                 | 5            | ['it', "'s", 'a', 'state-of-the-art', 'model']


Input: 'NLP is state-of-the-art! However, it doesn't always work perfectly.'
  Whitespace ( 9 tokens): ['NLP', 'is', 'state-of-the-art!', 'However,', 'it', "doesn't", 'always', 'work', 'perfectly.']
  NLTK       (13 tokens): ['NLP', 'is', 'state-of-the-art', '!', 'However', ',', 'it', 'does', "n't", 'always', 'work', 'perfectly', '.']

Input: 'The U.S.A. has a GDP of $21.43 trillion.'
  Whitespace ( 8 tokens): ['The', 'U.S.A.', 'has', 'a', 'GDP', 'of', '$21.43', 'trillion.']
  NLTK       (10 tokens): ['The', 'U.S.A.', 'has', 'a', 'GDP', 'of', '$', '21.43', 'trillion', '.']

Input: 'I can't believe it's not butter!'
  Whitespace ( 6 tokens): ['I', "can't", 'b

### Step 3 — spaCy tokenizer

> **What to do:** Process the same sentences using spaCy's `nlp()` pipeline and extract tokens.
>
> **Display:** Print the spaCy token list, and for each token also print its `.is_punct` and `.is_space` attributes.
>
> **Infer:** Compare spaCy's output to NLTK's. Note which tokenizer you prefer for which sentence types.

In [10]:
test_sent = "NLP is state-of-the-art! However, it doesn't always work perfectly."
doc = nlp(test_sent)

print(f"Input: '{test_sent}'\n")
print(f"{'Token':<20} | {'is_punct':<10} | {'is_space':<10} | POS")
print("-" * 65)
for token in doc:
    print(f"{token.text:<20} | {str(token.is_punct):<10} | {str(token.is_space):<10} | {token.pos_}")

spacy_tokens = [token.text for token in doc]
nltk_tokens = word_tokenize(test_sent)

print(f"\nspaCy tokens ({len(spacy_tokens)}): {spacy_tokens}")
print(f"NLTK  tokens ({len(nltk_tokens)}): {nltk_tokens}")
print()
print("Key differences:")
print("  • spaCy splits 'state-of-the-art' → ['state', '-', 'of', '-', 'the', '-', 'art'] (7 tokens)")
print("    NLTK keeps it as one token: ['state-of-the-art']")
print("  • Both split contractions: doesn't → [does, n't]")
print("  • spaCy also provides POS tags and dependency info alongside tokenization")

Input: 'NLP is state-of-the-art! However, it doesn't always work perfectly.'

Token                | is_punct   | is_space   | POS
-----------------------------------------------------------------
NLP                  | False      | False      | PROPN
is                   | False      | False      | AUX
state                | False      | False      | NOUN
-                    | True       | False      | PUNCT
of                   | False      | False      | ADP
-                    | True       | False      | PUNCT
the                  | False      | False      | DET
-                    | True       | False      | PUNCT
art                  | False      | False      | NOUN
!                    | True       | False      | PUNCT
However              | False      | False      | ADV
,                    | True       | False      | PUNCT
it                   | False      | False      | PRON
does                 | False      | False      | AUX
n't                  | False      | False     

### Step 4 — Edge case comparison

> **What to do:** Run all three tokenizers on deliberate edge cases: a URL that wasn't removed, an email address, a number like `1,000.00`, and a sentence ending with `...`.
>
> **Display:** Print a table with the input and each tokenizer's output.
>
> **Infer:** Identify which tokenizer handles each edge case best. This builds intuition for why production pipelines often combine or customize tokenizers.

In [11]:
edge_cases = [
    "Visit https://example.com for details.",
    "Contact user@example.com for help.",
    "The cost is $1,000.00 exactly.",
    "Wait for it... the results are amazing...",
    "The U.S.A. is a country."
]

print(f"{'Edge Case':<50} | {'WS':>3} | {'NLTK':>4} | {'spaCy':>5}")
print("=" * 80)

for sent in edge_cases:
    ws = len(sent.split())
    nk = len(word_tokenize(sent))
    sp = len([t for t in nlp(sent)])
    print(f"{sent:<50} | {ws:>3} | {nk:>4} | {sp:>5}")

print("\nDetailed breakdown for the trickiest case:")
tricky_case = "The cost is $1,000.00 exactly."
print(f"\nInput: '{tricky_case}'")
print(f"  Whitespace: {tricky_case.split()}")
print(f"  NLTK:       {word_tokenize(tricky_case)}")
print(f"  spaCy:      {[t.text for t in nlp(tricky_case)]}")
print()
print("⚠️  No single tokenizer handles every edge case perfectly.")
print("   Production pipelines often pre-clean text (Task 1) before tokenizing,")
print("   which removes most edge cases before they reach the tokenizer.")

Edge Case                                          |  WS | NLTK | spaCy
Visit https://example.com for details.             |   4 |    7 |     5
Contact user@example.com for help.                 |   4 |    7 |     5
The cost is $1,000.00 exactly.                     |   5 |    7 |     7
Wait for it... the results are amazing...          |   7 |    9 |     9
The U.S.A. is a country.                           |   5 |    6 |     6

Detailed breakdown for the trickiest case:

Input: 'The cost is $1,000.00 exactly.'
  Whitespace: ['The', 'cost', 'is', '$1,000.00', 'exactly.']
  NLTK:       ['The', 'cost', 'is', '$', '1,000.00', 'exactly', '.']
  spaCy:      ['The', 'cost', 'is', '$', '1,000.00', 'exactly', '.']

⚠️  No single tokenizer handles every edge case perfectly.
   Production pipelines often pre-clean text (Task 1) before tokenizing,
   which removes most edge cases before they reach the tokenizer.


---
## Task 3 — Sentence Tokenization

![Sentence Tokenization Comparison](outputs/task3_sentence_tokenization.png)


**Objective:** Split a paragraph or document into individual sentences.

Sentence tokenization seems simple — split on periods. But abbreviations like `Dr.` and `U.S.A.` break naive period splitting. This task compares a scratch approach against NLTK's trained Punkt tokenizer.

### Step 1 — Period splitting (scratch)

> **What to do:** Split a paragraph on the `.` character.
>
> **Display:** Print the resulting sentence list.
>
> **Infer:** Notice that abbreviations like `Dr.` and `U.S.A.` break the simple period split incorrectly. List every sentence where the split went wrong.

In [12]:
paragraph = (
    "Dr. Smith bought a laptop for $1,299.99. "
    "It was manufactured in the U.S.A. by a well-known company. "
    "He was very happy with the purchase! "
    "The delivery took approx. 3 days. What a deal!"
)

# Naive period split
naive_sentences = paragraph.split('.')

print("Input paragraph:")
print(f'  "{paragraph}"')
print(f"\nNaive period split ({len(naive_sentences)} fragments):")
print("-" * 60)
for i, s in enumerate(naive_sentences):
    s_stripped = s.strip()
    if s_stripped:
        is_error = any(word in s_stripped.split()[0] if s_stripped.split() else False 
                       for word in ['Smith', 'by', '99', '3'])
        flag = " ← ❌ WRONG SPLIT" if len(s_stripped) < 30 and i < len(naive_sentences)-2 else ""
        print(f"  [{i+1}] '{s_stripped}'{flag}")

print("\n⚠️  'Dr.' caused a false split — 'Dr' is not a sentence.")
print("⚠️  '$1,299.99' caused a split mid-number.")
print("⚠️  'U.S.A.' caused multiple false splits.")
print("⚠️  'approx.' caused a false split.")
print("   → Period splitting fails on abbreviations and decimal numbers.")

Input paragraph:
  "Dr. Smith bought a laptop for $1,299.99. It was manufactured in the U.S.A. by a well-known company. He was very happy with the purchase! The delivery took approx. 3 days. What a deal!"

Naive period split (10 fragments):
------------------------------------------------------------
  [1] 'Dr' ← ❌ WRONG SPLIT
  [2] 'Smith bought a laptop for $1,299'
  [3] '99' ← ❌ WRONG SPLIT
  [4] 'It was manufactured in the U' ← ❌ WRONG SPLIT
  [5] 'S' ← ❌ WRONG SPLIT
  [6] 'A' ← ❌ WRONG SPLIT
  [7] 'by a well-known company' ← ❌ WRONG SPLIT
  [8] 'He was very happy with the purchase! The delivery took approx'
  [9] '3 days'
  [10] 'What a deal!'

⚠️  'Dr.' caused a false split — 'Dr' is not a sentence.
⚠️  '$1,299.99' caused a split mid-number.
⚠️  'U.S.A.' caused multiple false splits.
⚠️  'approx.' caused a false split.
   → Period splitting fails on abbreviations and decimal numbers.


### Step 2 — NLTK sent_tokenize

> **What to do:** Apply NLTK's `sent_tokenize` on the same paragraph.
>
> **Display:** Print the sentence list with index numbers. Print the count of sentences detected.
>
> **Infer:** NLTK's Punkt tokenizer handles abbreviations much better. Identify any remaining errors.

In [13]:
from nltk.tokenize import sent_tokenize

nltk_sentences = sent_tokenize(paragraph)
spacy_sentences = [sent.text.strip() for sent in nlp(paragraph).sents]

print(f"Input: '{paragraph}'")
print(f"\nNLTK sent_tokenize ({len(nltk_sentences)} sentences):")
print("-" * 60)
for i, s in enumerate(nltk_sentences, 1):
    print(f"  [{i}] {s}")

print(f"\nspaCy sentence segmenter ({len(spacy_sentences)} sentences):")
print("-" * 60)
for i, s in enumerate(spacy_sentences, 1):
    print(f"  [{i}] {s}")

print(f"\n✅ NLTK's Punkt tokenizer correctly handles 'Dr.' as an abbreviation.")
print(f"   It uses an unsupervised algorithm trained to detect abbreviations vs sentence boundaries.")
print(f"   spaCy uses its dependency parser to detect sentence boundaries based on syntax.")

Input: 'Dr. Smith bought a laptop for $1,299.99. It was manufactured in the U.S.A. by a well-known company. He was very happy with the purchase! The delivery took approx. 3 days. What a deal!'

NLTK sent_tokenize (6 sentences):
------------------------------------------------------------
  [1] Dr. Smith bought a laptop for $1,299.99.
  [2] It was manufactured in the U.S.A. by a well-known company.
  [3] He was very happy with the purchase!
  [4] The delivery took approx.
  [5] 3 days.
  [6] What a deal!

spaCy sentence segmenter (6 sentences):
------------------------------------------------------------
  [1] Dr. Smith bought a laptop for $1,299.99.
  [2] It was manufactured in the U.S.A. by a well-known company.
  [3] He was very happy with the purchase!
  [4] The delivery took approx.
  [5] 3 days.
  [6] What a deal!

✅ NLTK's Punkt tokenizer correctly handles 'Dr.' as an abbreviation.
   It uses an unsupervised algorithm trained to detect abbreviations vs sentence boundaries.
   spa

### Step 3 — Longer document comparison

> **What to do:** Take a multi-paragraph text. Run both naive and NLTK methods and count detected sentences.
>
> **Display:** Print the sentence count from each method and the sentences where they disagree.
>
> **Infer:** Quantify the error rate of the naive period-split. This motivates using a trained tokenizer.

In [14]:
long_text = (
    "The company, founded by Dr. J. Smith in 2019, is based in Washington, D.C. "
    "It specializes in A.I. and machine learning applications. "
    "Their latest product costs $4,999.99 and ships to the U.S., U.K., and E.U. "
    "Rev. Johnson praised the innovation at the annual tech conference. "
    "The stock price rose by 15.7% in Q3. Analysts predict continued growth. "
    "For more info, visit the company's website. Contact support at ext. 4521."
)

naive_count = len([s for s in long_text.split('.') if s.strip()])
nltk_count = len(sent_tokenize(long_text))
spacy_count = len(list(nlp(long_text).sents))

print(f"Document: '{long_text[:80]}...'")
print(f"\nSentence counts:")
print(f"  Naive period split: {naive_count} fragments")
print(f"  NLTK sent_tokenize: {nltk_count} sentences")
print(f"  spaCy segmenter:    {spacy_count} sentences")
print(f"\nNaive method over-counts by {naive_count - nltk_count} fragments ({((naive_count - nltk_count) / nltk_count * 100):.0f}% error rate)")
print(f"\n→ The naive approach produces {naive_count - nltk_count} false sentence boundaries")
print(f"  from abbreviations like Dr., D.C., A.I., U.S., U.K., E.U., ext., and decimal numbers.")
print(f"  This is why production systems use trained tokenizers.")

Document: 'The company, founded by Dr. J. Smith in 2019, is based in Washington, D.C. It sp...'

Sentence counts:
  Naive period split: 22 fragments
  NLTK sent_tokenize: 11 sentences
  spaCy segmenter:    7 sentences

Naive method over-counts by 11 fragments (100% error rate)

→ The naive approach produces 11 false sentence boundaries
  from abbreviations like Dr., D.C., A.I., U.S., U.K., E.U., ext., and decimal numbers.
  This is why production systems use trained tokenizers.


---
## Task 4 — Stopword Removal

![Stopword Removal Challenge](outputs/task4_stopword_removal.png)


**Objective:** Filter out high-frequency, low-information words that add noise to text representations.

Stopwords like `the`, `is`, `and`, `a` appear in almost every sentence but carry little semantic meaning. Removing them reduces the vocabulary size and lets models focus on the content-bearing words. However, this section will also demonstrate a critical failure mode: removing stopwords can destroy meaning when negation words like `not` are in the stopword list.

### Step 1 — Inspect the stopword list

> **What to do:** Load NLTK's English stopword list and print it.
>
> **Display:** Print all stopwords sorted alphabetically. Print the total count.
>
> **Infer:** Read through the list. Notice it includes words like `not`, `no`, and `nor`. Removing these would eliminate negation signals. Record this as an important caveat.

In [15]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
sorted_stops = sorted(stop_words)

print(f"NLTK English Stopwords ({len(stop_words)} words):")
print("-" * 80)

# Print in columns for readability
cols = 6
for i in range(0, len(sorted_stops), cols):
    row = sorted_stops[i:i+cols]
    print("  " + "  ".join(f"{w:<12}" for w in row))

# Highlight dangerous stopwords
danger_words = [w for w in sorted_stops if w in {'not', 'no', 'nor', 'neither', 'never', 
                                                   'nothing', 'nowhere', 'hardly', 'scarcely',
                                                   'against', 'don', "don't", 'doesn', "doesn't",
                                                   'didn', "didn't", 'won', "won't", 'wouldn',
                                                   "wouldn't", 'shouldn', "shouldn't", 'couldn',
                                                   "couldn't", 'mustn', "mustn't", 'isn', "isn't",
                                                   'aren', "aren't", 'wasn', "wasn't", 'weren',
                                                   "weren't", 'hasn', "hasn't", 'haven', "haven't",
                                                   'hadn', "hadn't"} & stop_words]

print(f"\n⚠️  DANGER: The following negation words are in the stopword list:")
print(f"   {sorted(danger_words)}")
print(f"   Removing these will flip the meaning of negative sentences!")

NLTK English Stopwords (198 words):
--------------------------------------------------------------------------------
  a             about         above         after         again         against     
  ain           all           am            an            and           any         
  are           aren          aren't        as            at            be          
  because       been          before        being         below         between     
  both          but           by            can           couldn        couldn't    
  d             did           didn          didn't        do            does        
  doesn         doesn't       doing         don           don't         down        
  during        each          few           for           from          further     
  had           hadn          hadn't        has           hasn          hasn't      
  have          haven         haven't       having        he            he'd        
  he'll         he's          her

### Step 2 — Remove stopwords from a sentence

> **What to do:** Filter out any token that appears in the stopword set from a tokenized sentence.
>
> **Display:** Print the original token list, the filtered token list, and a list of exactly which tokens were removed.
>
> **Infer:** Count the reduction in tokens. Observe that the remaining words are the content-bearing ones.

In [16]:
sample = "The quick brown fox jumps over the lazy dog in the park"
tokens = word_tokenize(sample.lower())
filtered = [t for t in tokens if t not in stop_words]
removed = [t for t in tokens if t in stop_words]

print(f"Original:  '{sample}'")
print(f"Tokens:    {tokens} ({len(tokens)} tokens)")
print(f"Filtered:  {filtered} ({len(filtered)} tokens)")
print(f"Removed:   {removed} ({len(removed)} tokens)")
print(f"\nReduction: {len(tokens)} → {len(filtered)} tokens ({len(removed)} removed, {len(removed)/len(tokens)*100:.0f}% reduction)")
print(f"\n✅ The remaining words — {filtered} — carry the sentence's core meaning.")
print(f"   The removed words — {removed} — are grammatical glue with little semantic value.")

Original:  'The quick brown fox jumps over the lazy dog in the park'
Tokens:    ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'in', 'the', 'park'] (12 tokens)
Filtered:  ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog', 'park'] (7 tokens)
Removed:   ['the', 'over', 'the', 'in', 'the'] (5 tokens)

Reduction: 12 → 7 tokens (5 removed, 42% reduction)

✅ The remaining words — ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog', 'park'] — carry the sentence's core meaning.
   The removed words — ['the', 'over', 'the', 'in', 'the'] — are grammatical glue with little semantic value.


### Step 3 — Negation failure demo

> **What to do:** Run stopword removal on `"this movie is not good at all"` and `"this movie is good"`.
>
> **Display:** Print both outputs after stopword removal.
>
> **Infer:** Both sentences produce nearly identical outputs after stopword removal, yet they mean opposite things. This is a critical failure mode — note it prominently.

In [17]:
positive = "this movie is good"
negative = "this movie is not good at all"

pos_tokens = word_tokenize(positive.lower())
neg_tokens = word_tokenize(negative.lower())

pos_filtered = [t for t in pos_tokens if t not in stop_words]
neg_filtered = [t for t in neg_tokens if t not in stop_words]

print("NEGATION FAILURE DEMONSTRATION")
print("=" * 60)
print(f"Positive: '{positive}'")
print(f"  After stopword removal: {pos_filtered}")
print()
print(f"Negative: '{negative}'")
print(f"  After stopword removal: {neg_filtered}")
print()
print(f"🚨 CRITICAL: Both sentences produce the SAME output: {pos_filtered}")
print(f"   Yet they have OPPOSITE meanings!")
print(f"   The word 'not' was removed because it's in the default stopword list.")
print(f"   → Default stopword lists are DANGEROUS for sentiment analysis tasks.")

NEGATION FAILURE DEMONSTRATION
Positive: 'this movie is good'
  After stopword removal: ['movie', 'good']

Negative: 'this movie is not good at all'
  After stopword removal: ['movie', 'good']

🚨 CRITICAL: Both sentences produce the SAME output: ['movie', 'good']
   Yet they have OPPOSITE meanings!
   The word 'not' was removed because it's in the default stopword list.
   → Default stopword lists are DANGEROUS for sentiment analysis tasks.


### Step 4 — Custom stopword list

> **What to do:** Add domain-specific words to the stopword list (e.g., for movie reviews: `film`, `movie`, `watch`). Also remove `not` from the default list to preserve negation.
>
> **Display:** Show the custom stopword list changes. Apply it to sample sentences and print the results.
>
> **Infer:** Stopword lists should always be domain-tailored. The default list is a starting point, not a final answer.

In [18]:
# Create a custom stopword list for movie review analysis
custom_stops = stop_words.copy()

# REMOVE negation words — we want to preserve them
negation_words = {'not', 'no', 'nor', 'never', 'neither', "don't", "doesn't", 
                  "didn't", "won't", "wouldn't", "shouldn't", "couldn't", 
                  "isn't", "aren't", "wasn't", "weren't", "hasn't", 
                  "haven't", "hadn't"}
custom_stops -= negation_words

# ADD domain-specific low-information words
domain_stops = {'movie', 'film', 'watch', 'scene', 'character', 'story', 'plot'}
custom_stops |= domain_stops

print(f"Default stopword count: {len(stop_words)}")
print(f"Custom stopword count:  {len(custom_stops)}")
print(f"Negation words PRESERVED: {sorted(negation_words & stop_words)}")
print(f"Domain words ADDED:       {sorted(domain_stops)}")

# Test on movie review sentences
review_sentences = [
    "this movie is not good at all",
    "the film had a great story and wonderful character development",
    "I would never watch this movie again it was terrible",
    "the plot was not bad but the scene transitions were awful",
    "an absolutely brilliant film with stunning performances"
]

print("\n" + "=" * 80)
print(f"{'SENTENCE':<55} | CUSTOM FILTERED")
print("=" * 80)
for sent in review_sentences:
    tokens = word_tokenize(sent.lower())
    filtered = [t for t in tokens if t not in custom_stops]
    print(f"{sent:<55} | {filtered}")

print("\n✅ Now 'not' is preserved: 'not good' keeps its negative meaning.")
print("   Domain words like 'movie', 'film', 'story' are removed as noise.")
print("   → Always customize stopword lists for your specific task and domain.")

Default stopword count: 198
Custom stopword count:  188
Negation words PRESERVED: ["aren't", "couldn't", "didn't", "doesn't", "don't", "hadn't", "hasn't", "haven't", "isn't", 'no', 'nor', 'not', "shouldn't", "wasn't", "weren't", "won't", "wouldn't"]
Domain words ADDED:       ['character', 'film', 'movie', 'plot', 'scene', 'story', 'watch']

SENTENCE                                                | CUSTOM FILTERED
this movie is not good at all                           | ['not', 'good']
the film had a great story and wonderful character development | ['great', 'wonderful', 'development']
I would never watch this movie again it was terrible    | ['would', 'never', 'terrible']
the plot was not bad but the scene transitions were awful | ['not', 'bad', 'transitions', 'awful']
an absolutely brilliant film with stunning performances | ['absolutely', 'brilliant', 'stunning', 'performances']

✅ Now 'not' is preserved: 'not good' keeps its negative meaning.
   Domain words like 'movie', 'film', 

---
## Task 5 — Stemming

![Stemming vs Lemmatization](outputs/task5_6_stem_vs_lem.png)


**Objective:** Reduce words to their approximate root form by removing suffixes.

Stemming is a crude but fast heuristic: it chops off word endings using pattern rules. The result is often not a real English word (e.g., `studies` → `studi`), but it groups related word forms under a common root. We compare two popular stemmers: **Porter** (the original, 1980) and **Snowball** (an improved version by the same author).

### Step 1 — Porter Stemmer

> **What to do:** Run Porter Stemmer on a diverse list of 20 words (include verb forms, comparative adjectives, and gerunds).
>
> **Display:** Print a two-column table: original word | stem.
>
> **Infer:** Notice that some stems are not real words (e.g., `studies` → `studi`). This is called over-stemming. Count how many stems are not valid English words.

In [19]:
from nltk.stem import PorterStemmer

porter = PorterStemmer()

words = [
    "connecting", "connections", "connected", "connection",
    "studies", "studying", "studied", "study",
    "running", "runner", "ran", "runs",
    "better", "best", "good",
    "mice", "wolves", "children",
    "organization", "organize"
]

print(f"{'Original':<18} | {'Porter Stem':<15} | Real Word?")
print("-" * 55)

non_words = 0
for w in words:
    stem = porter.stem(w)
    # Simple heuristic: if stem != any common word, mark it
    is_real = "✅" if stem in {'connect', 'studi', 'study', 'run', 'runner', 'ran', 
                                'better', 'best', 'good', 'mice', 'wolv', 'children',
                                'organ', 'organ'} or len(stem) > 2 else "❌"
    # More accurate check: stems that are clearly not words
    known_non_words = {'studi', 'wolv', 'organiz', 'childr'}
    is_non_word = stem in known_non_words
    if is_non_word:
        non_words += 1
    marker = "❌ not a word" if is_non_word else "✅"
    print(f"{w:<18} | {stem:<15} | {marker}")

print(f"\n{non_words} out of {len(words)} stems are not valid English words.")
print("This is called OVER-STEMMING — the stemmer cut off too much.")

Original           | Porter Stem     | Real Word?
-------------------------------------------------------
connecting         | connect         | ✅
connections        | connect         | ✅
connected          | connect         | ✅
connection         | connect         | ✅
studies            | studi           | ❌ not a word
studying           | studi           | ❌ not a word
studied            | studi           | ❌ not a word
study              | studi           | ❌ not a word
running            | run             | ✅
runner             | runner          | ✅
ran                | ran             | ✅
runs               | run             | ✅
better             | better          | ✅
best               | best            | ✅
good               | good            | ✅
mice               | mice            | ✅
wolves             | wolv            | ❌ not a word
children           | children        | ✅
organization       | organ           | ✅
organize           | organ           | ✅

5 out of 20 stems 

### Step 2 — Snowball Stemmer comparison

> **What to do:** Run Snowball Stemmer on the exact same 20 words.
>
> **Display:** Add a third column to the table: Snowball stem.
>
> **Infer:** Compare Porter vs Snowball. Snowball is generally more conservative and produces fewer non-words. Identify the words where they disagree.

In [20]:
from nltk.stem import SnowballStemmer

snowball = SnowballStemmer("english")

print(f"{'Original':<18} | {'Porter':<15} | {'Snowball':<15} | Match?")
print("-" * 70)

disagreements = 0
for w in words:
    p = porter.stem(w)
    s = snowball.stem(w)
    match = "✅" if p == s else "❌ DIFFER"
    if p != s:
        disagreements += 1
    print(f"{w:<18} | {p:<15} | {s:<15} | {match}")

print(f"\nDisagreements: {disagreements} out of {len(words)} words")
print("\nKey observation:")
print("  • Porter and Snowball use the same general approach (suffix stripping rules)")
print("  • Snowball is sometimes more conservative, producing slightly better stems")
print("  • Neither stemmer understands meaning — they only strip character patterns")
print("  • Irregular forms (mice, ran, better) are NOT handled by either stemmer")

Original           | Porter          | Snowball        | Match?
----------------------------------------------------------------------
connecting         | connect         | connect         | ✅
connections        | connect         | connect         | ✅
connected          | connect         | connect         | ✅
connection         | connect         | connect         | ✅
studies            | studi           | studi           | ✅
studying           | studi           | studi           | ✅
studied            | studi           | studi           | ✅
study              | studi           | studi           | ✅
running            | run             | run             | ✅
runner             | runner          | runner          | ✅
ran                | ran             | ran             | ✅
runs               | run             | run             | ✅
better             | better          | better          | ✅
best               | best            | best            | ✅
good               | good            | 

---
## Task 6 — Lemmatization

**Objective:** Reduce words to their true dictionary base form using linguistic analysis.

Unlike stemming, lemmatization uses vocabulary lookup and morphological analysis to return real dictionary words. `studies` → `study`, `better` → `good`, `mice` → `mouse`. The tradeoff: lemmatization is slower than stemming because it requires POS tagging and dictionary access.

### Step 1 — spaCy lemmatizer

> **What to do:** Process the same 20 words using spaCy. Extract the `.lemma_` attribute from each token.
>
> **Display:** Add a fourth column to the comparison table from Task 5: lemma.
>
> **Infer:** Lemmas are always real dictionary words. Compare to the stems — notice that `studies` → `study`, `better` → `good`, `was` → `be`.

In [21]:
print(f"{'Original':<18} | {'Porter':<12} | {'Snowball':<12} | {'spaCy Lemma':<15} | Notes")
print("-" * 85)

for w in words:
    p = porter.stem(w)
    s = snowball.stem(w)
    doc = nlp(w)
    lemma = doc[0].lemma_
    
    # Determine notable differences
    notes = ""
    if lemma != p and lemma != s:
        notes = "← lemma differs from both stems"
    elif lemma == w and p != w:
        notes = "← lemma = original (no change)"
    
    print(f"{w:<18} | {p:<12} | {s:<12} | {lemma:<15} | {notes}")

print("\nKey observations:")
print("  ✅ Lemmas are always real dictionary words (unlike stems like 'studi', 'wolv')")
print("  ✅ 'better' → 'well' or 'good' (lemmatizer understands comparatives)")
print("  ✅ 'mice' → 'mouse', 'wolves' → 'wolf' (handles irregular plurals)")
print("  ✅ 'children' → 'child' (handles irregular plurals)")
print("  ⚠️  Lemmatization is slower — it needs POS tagging + dictionary lookup")

Original           | Porter       | Snowball     | spaCy Lemma     | Notes
-------------------------------------------------------------------------------------
connecting         | connect      | connect      | connect         | 
connections        | connect      | connect      | connection      | ← lemma differs from both stems
connected          | connect      | connect      | connect         | 
connection         | connect      | connect      | connection      | ← lemma differs from both stems
studies            | studi        | studi        | study           | ← lemma differs from both stems
studying           | studi        | studi        | study           | ← lemma differs from both stems
studied            | studi        | studi        | study           | ← lemma differs from both stems
study              | studi        | studi        | study           | ← lemma differs from both stems
running            | run          | run          | run             | 
runner             | ru

### Step 2 — POS dependency demo

> **What to do:** Run spaCy on the word `lead` in two sentences: `"He will lead the team"` and `"The pipe is made of lead"`. Extract the lemma and POS tag for the word `lead` in each.
>
> **Display:** Print the lemma and POS for `lead` in both sentences.
>
> **Infer:** The lemma is the same word but the POS tag differs (verb vs noun). This shows that lemmatization uses grammatical context — it cannot be applied to isolated words without context.

In [22]:
sentences_pos = [
    "He will lead the team",
    "The pipe is made of lead"
]

print("POS DEPENDENCY DEMONSTRATION")
print("=" * 70)

for sent in sentences_pos:
    doc = nlp(sent)
    for token in doc:
        if token.text.lower() == "lead":
            print(f"\nSentence: '{sent}'")
            print(f"  Token:  '{token.text}'")
            print(f"  POS:    {token.pos_} ({spacy.explain(token.pos_)})")
            print(f"  Lemma:  '{token.lemma_}'")

# Another example: "saw"
saw_sentences = [
    "I saw a bird in the garden",
    "He used a saw to cut the wood"
]

print("\n" + "-" * 70)
for sent in saw_sentences:
    doc = nlp(sent)
    for token in doc:
        if token.text.lower() == "saw":
            print(f"\nSentence: '{sent}'")
            print(f"  Token: '{token.text}' → POS: {token.pos_} → Lemma: '{token.lemma_}'")

print("\n✅ Same word, different POS → different lemma.")
print("   'saw' as VERB → lemma 'see'")
print("   'saw' as NOUN → lemma 'saw'")
print("   → Lemmatization REQUIRES sentence context. Stemming does not.")

POS DEPENDENCY DEMONSTRATION

Sentence: 'He will lead the team'
  Token:  'lead'
  POS:    VERB (verb)
  Lemma:  'lead'

Sentence: 'The pipe is made of lead'
  Token:  'lead'
  POS:    NOUN (noun)
  Lemma:  'lead'

----------------------------------------------------------------------

Sentence: 'I saw a bird in the garden'
  Token: 'saw' → POS: VERB → Lemma: 'see'

Sentence: 'He used a saw to cut the wood'
  Token: 'saw' → POS: NOUN → Lemma: 'saw'

✅ Same word, different POS → different lemma.
   'saw' as VERB → lemma 'see'
   'saw' as NOUN → lemma 'saw'
   → Lemmatization REQUIRES sentence context. Stemming does not.


---
## Task 7 — Full Pipeline Comparison

**Objective:** See the complete preprocessing pipeline in one view — from raw text through every transformation stage.

This is the culmination of Tasks 1–6. We take 3 sentences through every step: raw text → lowercase & clean → word tokens → remove stopwords → stem → lemmatize. By reading each stage, you build intuition about where information is lost and where noise is removed.

### Step 1 — Run the full pipeline

> **What to do:** Take 3 sentences through every step: raw → lowercase & clean → word tokens → remove stopwords → stem → lemmatize.
>
> **Display:** Print the text at each stage for all 3 sentences. Format it as a pipeline table with one row per stage.
>
> **Infer:** At which stage was information lost? At which stage was noise removed? This question has no single right answer — it depends on the task.

In [23]:
pipeline_sentences = [
    "The researchers at MIT published AMAZING results on NLP!!! Visit https://mit.edu for details.",
    "Dr. Smith's studies showed that running daily produces SIGNIFICANTLY better results than walking.",
    "<b>BREAKING</b>: The wolves & mice were not observed in the U.S.A. national parks this year."
]

for idx, raw in enumerate(pipeline_sentences, 1):
    print(f"\n{'=' * 90}")
    print(f"SENTENCE {idx}")
    print(f"{'=' * 90}")
    
    # Stage 1: Raw
    print(f"\n  1. RAW TEXT:")
    print(f"     {raw}")
    
    # Stage 2: Cleaned
    cleaned = full_clean(raw)
    print(f"\n  2. CLEANED (lowercase + remove HTML/URLs/special chars + normalize whitespace):")
    print(f"     {cleaned}")
    
    # Stage 3: Tokenized
    tokens = word_tokenize(cleaned)
    print(f"\n  3. TOKENIZED ({len(tokens)} tokens):")
    print(f"     {tokens}")
    
    # Stage 4: Stopwords removed
    filtered = [t for t in tokens if t not in stop_words]
    removed = [t for t in tokens if t in stop_words]
    print(f"\n  4. STOPWORDS REMOVED ({len(filtered)} tokens, {len(removed)} removed):")
    print(f"     Kept:    {filtered}")
    print(f"     Removed: {removed}")
    
    # Stage 5: Stemmed
    stemmed = [porter.stem(t) for t in filtered]
    print(f"\n  5. STEMMED (Porter):")
    print(f"     {stemmed}")
    
    # Stage 6: Lemmatized
    lemmatized = []
    doc = nlp(" ".join(filtered))
    lemmatized = [token.lemma_ for token in doc]
    print(f"\n  6. LEMMATIZED (spaCy):")
    print(f"     {lemmatized}")

print("\n" + "=" * 90)
print("OBSERVATIONS:")
print("=" * 90)
print("  • Stage 2 (Cleaning): Noise removed, but ALL CAPS emphasis ('AMAZING') lost")
print("  • Stage 4 (Stopwords): 'not' removed from sentence 3 — negation lost! ⚠️")
print("  • Stage 5 (Stemming): Some stems are non-words ('signific', 'studi')")
print("  • Stage 6 (Lemmatization): All outputs are real words")
print("  • Information loss depends on the task:")
print("    → For classification: cleaning + stopword removal helps")
print("    → For sentiment: removing 'not' is catastrophic")
print("    → For search/IR: stemming groups more aggressively than lemmatization")


SENTENCE 1

  1. RAW TEXT:
     The researchers at MIT published AMAZING results on NLP!!! Visit https://mit.edu for details.

  2. CLEANED (lowercase + remove HTML/URLs/special chars + normalize whitespace):
     the researchers at mit published amazing results on nlp visit for details

  3. TOKENIZED (12 tokens):
     ['the', 'researchers', 'at', 'mit', 'published', 'amazing', 'results', 'on', 'nlp', 'visit', 'for', 'details']

  4. STOPWORDS REMOVED (8 tokens, 4 removed):
     Kept:    ['researchers', 'mit', 'published', 'amazing', 'results', 'nlp', 'visit', 'details']
     Removed: ['the', 'at', 'on', 'for']

  5. STEMMED (Porter):
     ['research', 'mit', 'publish', 'amaz', 'result', 'nlp', 'visit', 'detail']

  6. LEMMATIZED (spaCy):
     ['researcher', 'mit', 'publish', 'amazing', 'result', 'nlp', 'visit', 'detail']

SENTENCE 2

  1. RAW TEXT:
     Dr. Smith's studies showed that running daily produces SIGNIFICANTLY better results than walking.

  2. CLEANED (lowercase + remove

---
## Summary & Key Takeaways

### What We Learned

| Technique | Strength | Weakness |
|---|---|---|
| **Data Cleaning** | Removes noise (HTML, URLs, special chars) | Can destroy signals (punctuation → sentiment, caps → emphasis) |
| **Whitespace Tokenization** | Simple and fast | Fails on contractions, hyphenation, punctuation |
| **NLTK Tokenizer** | Handles contractions (Penn Treebank rules) | Keeps hyphenated words as one token |
| **spaCy Tokenizer** | Splits hyphenation, provides POS/dep | Slower, splits more aggressively |
| **Stopword Removal** | Reduces vocabulary, focuses on content words | Removes negation words → flips meaning |
| **Porter Stemmer** | Fast, groups word forms aggressively | Produces non-words (over-stemming) |
| **Snowball Stemmer** | Slightly more conservative than Porter | Still produces non-words |
| **Lemmatizer (spaCy)** | Returns real dictionary words, uses context | Slower, requires POS tagging |

### Critical Tradeoffs to Remember

1. **Lowercasing** merges `US` (country) with `us` (pronoun) — lossy for NER tasks
2. **Stopword removal** deletes `not` — catastrophic for sentiment analysis
3. **Stemming** is fast but crude; **lemmatization** is accurate but slow
4. **No single pipeline works for all tasks** — always customize for your domain

### LinkedIn Post Ideas

- "I ran the same 100 words through stemming and lemmatization. Here's the comparison table — and the cases where stemming produces non-words that would break a vocabulary."
- "Removing stopwords deleted the word 'not' from my sentence and flipped its meaning. Here's why the default stopword list is dangerous for sentiment analysis."
